In [ ]:
import pdfplumber
import re

def read_pdf(fp):
    txt = ""
    fp = str(fp)
     
    try:
        with pdfplumber.open(fp) as pdf:
            for pg in pdf.pages:
                try:
                    page_text = pg.extract_text() or ""
                except Exception:
                    page_text = ""
                
                if not page_text.strip():
                    try:
                        page_text = pg.extract_text(x_tolerance=2) or ""
                    except Exception:
                        page_text = ""
                
                txt += page_text + " "
                
    except Exception:
        try:
            from pdfminer.high_level import extract_text as pm_extract_text
            txt = pm_extract_text(fp) or ""
        except Exception:
            txt = ""

    # clean text
    txt = re.sub(r'\s+', ' ', txt).strip()
    
    return txt

In [18]:
print(read_pdf("../dataset/JDs/jd1.pdf"))
jdtext=read_pdf("../dataset/JDs/jd1.pdf")


Job Title: Lead Data Engineer Experience: 8+ Years Location: Ahmedabad Job Type: Full-Time Department: Data & Analytics / Engineering About Us: Kenexai delivers smart, data-driven solutions that empower businesses across industries. Our mission is to combine deep, domain-specific expertise with cutting-edge technology to drive meaningful impact. With a trusted team, consistent quality, and a growing global presence, we remain committed to delivering excellence while staying true to our core values: innovation, integrity, and client success. Be part of a team that’s not just building solutions but shaping the future with intelligence. Role Overview: We are looking for an experienced and strategic Lead Data Engineer to oversee the design, development, and implementation of our enterprise data platform. As a technical leader, you will work across engineering, analytics, and business teams to deliver scalable data solutions, while also guiding and mentoring a team of data engineers. This i

In [21]:
import os
import logging

logging.getLogger("pdfminer").setLevel(logging.ERROR)

resume_texts = []

resume_folder = "../dataset/CVs"

for file in os.listdir(resume_folder):
    if file.endswith(".pdf"):
        path = os.path.join(resume_folder, file)
        text = read_pdf(path)
        resume_texts.append((file, text))

print("Total resumes:", len(resume_texts))

Total resumes: 13


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine JD + all resumes
documents = [jdtext] + [text for _, text in resume_texts]

vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = vectorizer.fit_transform(documents)

print("Matrix shape:", tfidf_matrix.shape)

Matrix shape: (14, 1750)


In [23]:
from sklearn.metrics.pairwise import cosine_similarity

# JD vector
jd_vector = tfidf_matrix[0]

# Resume vectors
resume_vectors = tfidf_matrix[1:]

# Similarity scores
scores = cosine_similarity(jd_vector, resume_vectors)

print(scores)

[[0.33288753 0.26886639 0.09172683 0.31127619 0.2659741  0.33259913
  0.32955529 0.3594056  0.3594056  0.31432228 0.30496136 0.3053688
  0.37305808]]


In [24]:
results = []

for i, score in enumerate(scores[0]):
    resume_name = resume_texts[i][0]
    results.append((resume_name, score))

# sort by highest score
results.sort(key=lambda x: x[1], reverse=True)

for r in results:
    print(r)

('Tanaji More.pdf', np.float64(0.37305808316989203))
('Rohit_Gaddamidi (1).pdf', np.float64(0.35940560212440703))
('Rohit_Gaddamidi.pdf', np.float64(0.35940560212440703))
('Abhijeet Sahu Resume_word.pdf', np.float64(0.3328875349152633))
('Golla Kanya Dharani Resume .pdf', np.float64(0.33259913110911477))
('Resume_AnshuSDE.pdf', np.float64(0.3295552939307818))
('Sachin T.pdf', np.float64(0.3143222758355575))
('Bhanu_Priya_Snowflake_Developer_ (1).pdf', np.float64(0.3112761880751474))
('Saumya - Data Engineer (1).pdf', np.float64(0.3053688001232404))
('Sai M_DE_Resume.pdf', np.float64(0.3049613561978684))
('Aluri_Nishanth_Azure_Data_Engineer_Updated (1) (1).pdf', np.float64(0.26886638931805046))
('Dhiraj Baldewa.pdf', np.float64(0.2659740968075944))
('Anuj-jani-DE.pdf', np.float64(0.0917268295374173))
